In [ ]:
import numpy as np
import pandas as pd

# 1. Generate & Save a Deliberately Messy Dataset (100 Rows)
np.random.seed(42)
raw_names = [
    '  aArAv sharma ',
    'riya VERMA',
    '  Kavya ',
    'rahul  ',
    np.nan,
    'SNEHA patel',
    'deepak',
    ' Tanvi   ',
]
raw_cities = [
    '  mumbai  ',
    'DELHI',
    'bengaluru',
    np.nan,
    ' pune ',
    'CHENNAI ',
    'hyd',
    'Hyderabad',
]

rows = []
for i in range(1, 101):
    rows.append({
        'customer_id': 1000 + i if i % 15 != 0 else np.nan,
        'name': np.random.choice(raw_names),
        'age': (
            str(np.random.randint(18, 65))
            if i % 10 != 0
            else np.random.choice(['twenty', np.nan])
        ),
        'city': np.random.choice(raw_cities),
        'email': (
            f'user_{i}@example.com' if i % 8 != 0 else f'USER_{i}@EXAMPLE.COM '
        ),
        'purchase_amount': (
            str(np.random.randint(100, 5000)) if i % 12 != 0 else np.nan
        ),
        'rating': np.random.choice([1, 2, 3, 4, 5, np.nan]),
    })

In [ ]:
# Add explicit duplicates
messy_df = pd.DataFrame(rows)
messy_df = pd.concat(
    [messy_df, messy_df.iloc[:8]], ignore_index=True
)  # duplicate 8 rows
messy_df.to_csv('messy_customer_data.csv', index=False)

In [ ]:
# 2. Data Cleaning Operations
df_dirty = pd.read_csv('messy_customer_data.csv')
initial_shape = df_dirty.shape
print(f'Shape Before Cleaning: {initial_shape}')

In [ ]:
# Missing Data Detection
print('\nMissing Values Per Column:\n', df_dirty.isnull().sum())

# Deduplication
print(f'\nDuplicate Rows: {df_dirty.duplicated().sum()}')
df_cleaned = df_dirty.drop_duplicates()

# String Sanitization (Names & Cities)
df_cleaned['name'] = df_cleaned['name'].astype(str).str.strip().str.title()
df_cleaned['name'] = df_cleaned['name'].replace('Nan', np.nan)

df_cleaned['city'] = df_cleaned['city'].astype(str).str.strip().str.title()
df_cleaned['city'] = df_cleaned['city'].replace(
    {'Hyd': 'Hyderabad', 'Nan': np.nan}
)

df_cleaned['email'] = df_cleaned['email'].astype(str).str.strip().str.lower()

In [ ]:
# Type Conversion & Imputation
# Age: remove invalid non-numeric strings, convert to numeric, impute with median
df_cleaned['age'] = pd.to_numeric(df_cleaned['age'], errors='coerce')
df_cleaned['age'] = (
    df_cleaned['age'].fillna(df_cleaned['age'].median()).astype(int)
)

# Purchase Amount: convert to numeric, impute with 0.0
df_cleaned['purchase_amount'] = pd.to_numeric(
    df_cleaned['purchase_amount'], errors='coerce'
)
df_cleaned['purchase_amount'] = df_cleaned['purchase_amount'].fillna(0.0)

# Rating: impute missing ratings with the column mode
df_cleaned['rating'] = df_cleaned['rating'].fillna(
    df_cleaned['rating'].mode()[0]
)

# Customer ID: drop remaining null ID records
df_cleaned = df_cleaned.dropna(subset=['customer_id'])
df_cleaned['customer_id'] = df_cleaned['customer_id'].astype(int)

# Shape Comparison
print(f'Shape After Cleaning: {df_cleaned.shape}')
print(f'Total records removed: {initial_shape[0] - df_cleaned.shape[0]}')

In [ ]:
# Export Clean Dataset
df_cleaned.to_csv('cleaned_customer_data.csv', index=False)